# Phase 3-5: run the pre-registration ACCOUNT loads

Drives dry-runs, probes, the bulk update (the InvestCustomer__pc flag on 1,696 existing accounts),
the bulk insert (1,495 new person accounts), the id writebacks, and the CPE backfill.
Consents are `05_run_prereg_consents.ipynb`.

Safety model (as in the August invest-consent run):
- every load goes through the loader scripts (`subprocess`, non-zero return = stop);
- probes are single REST calls against ONE business-approved record, gated by
  `RUN_PROBE_*` flags plus a hand-entered id/external_id;
- the bulk loads are gated by `RUN_LOAD_*` flags — **they stay False until Arsal's
  explicit go-ahead**;
- `EXPECTED` comes verbatim from the 03 staging contract; any drift aborts.

In [1]:
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root: config, mysql_client, sf client
sys.path.insert(0, str(Path.cwd()))          # this job's loaders

from config import load_mysql_config
from mysql_client import MySQLClient

REPO_ROOT = Path.cwd().parent
BATCH_DATE = "2026-08-28"
UPDATE_BATCHES = [f"{BATCH_DATE}_prereg_update_optin", f"{BATCH_DATE}_prereg_update_optout"]
INSERT_BATCHES = [f"{BATCH_DATE}_prereg_insert_optin", f"{BATCH_DATE}_prereg_insert_optout"]

# Staging contract, frozen 2026-08-28 (03 section 8): 3,192 rows total,
# 1,696 updates, 1,496 insert rows of which 1 excluded (Daurer, shared mailbox).
# Loadable counts, i.e. staged minus excluded:
EXPECTED = {
    f"{BATCH_DATE}_prereg_update_optin":  1696,
    f"{BATCH_DATE}_prereg_update_optout": 0,
    f"{BATCH_DATE}_prereg_insert_optin":  1495,
    f"{BATCH_DATE}_prereg_insert_optout": 0,
}

db = MySQLClient(load_mysql_config())
print("connected")

connected


## 1. Reload with the loaders' own predicates

Counts must equal the 03 contract exactly (loadable = staged minus excluded);
anything else means the staging table moved since the freeze.

In [2]:
for batch_id in UPDATE_BATCHES:
    n = db.fetch_one("""
        SELECT COUNT(*) AS n FROM crm_imp_person_accounts
        WHERE _batch_id = %s AND _operation = 'update' AND _excluded = 0
          AND _processed_at IS NULL AND sf_account_id IS NOT NULL
    """, (batch_id,))["n"]
    print(f"{batch_id}: {n:,} loadable (expected {EXPECTED[batch_id]:,})")
    assert n == EXPECTED[batch_id], f"drift on {batch_id}"

for batch_id in INSERT_BATCHES:
    n = db.fetch_one("""
        SELECT COUNT(*) AS n FROM crm_imp_person_accounts
        WHERE _batch_id = %s AND _operation = 'insert' AND _excluded = 0
          AND _account_processed_at IS NULL
    """, (batch_id,))["n"]
    print(f"{batch_id}: {n:,} loadable (expected {EXPECTED[batch_id]:,})")
    assert n == EXPECTED[batch_id], f"drift on {batch_id}"

print("reload contract holds")

2026-08-28_prereg_update_optin: 1,696 loadable (expected 1,696)
2026-08-28_prereg_update_optout: 0 loadable (expected 0)
2026-08-28_prereg_insert_optin: 1,495 loadable (expected 1,495)
2026-08-28_prereg_insert_optout: 0 loadable (expected 0)
reload contract holds


## 2. Dry-runs (no Salesforce contact)

Writes the exact Bulk CSVs to `local_data/dry_run_prereg_*.csv`. Eyeball them:
the update CSV must have ONLY `Id, InvestCustomer__pc` (prospects: no
investment status or expiration); insert CSVs must carry `ExternalID__pc` first and no
consent columns.

In [3]:
def run_loader(script: str, batch_id: str, dry_run: bool = False) -> None:
    cmd = [sys.executable, str(Path.cwd() / script), batch_id] + (["--dry-run"] if dry_run else [])
    print(">>", " ".join(cmd[1:]))
    res = subprocess.run(cmd, cwd=REPO_ROOT)
    assert res.returncode == 0, f"{script} {batch_id} exited {res.returncode}"

for b in UPDATE_BATCHES:
    run_loader("update_accounts_prereg.py", b, dry_run=True)
for b in INSERT_BATCHES:
    run_loader("insert_person_accounts_prereg.py", b, dry_run=True)

>> d:\repos\dev\Arsal\bi-crm-imports\invest-preregistration\update_accounts_prereg.py 2026-08-28_prereg_update_optin --dry-run
>> d:\repos\dev\Arsal\bi-crm-imports\invest-preregistration\update_accounts_prereg.py 2026-08-28_prereg_update_optout --dry-run
>> d:\repos\dev\Arsal\bi-crm-imports\invest-preregistration\insert_person_accounts_prereg.py 2026-08-28_prereg_insert_optin --dry-run
>> d:\repos\dev\Arsal\bi-crm-imports\invest-preregistration\insert_person_accounts_prereg.py 2026-08-28_prereg_insert_optout --dry-run


## 3. Probe: ONE account update

Business-approved account from the optin update batch. Single REST update of the
InvestCustomer__pc flag, then readback (the readback also shows
InvestmentStatus__pc/ExpirationDate to prove they stayed untouched). Idempotent under the later bulk load (same
values written twice). Set `PROBE_UPDATE_ACCOUNT_ID` by hand — no auto-pick.

In [5]:
RUN_PROBE_UPDATE = True
PROBE_UPDATE_ACCOUNT_ID = "001Te00000ZqiFLIAZ"   # hand-entered, business-agreed

if RUN_PROBE_UPDATE:
    from salesforce_client_prod import SalesforceClientCC, load_salesforce_cc_config_from_env
    from update_accounts_prereg import row_to_sf_record

    row = db.fetch_one("""
        SELECT * FROM crm_imp_person_accounts
        WHERE _batch_id = %s AND sf_account_id = %s
    """, (UPDATE_BATCHES[0], PROBE_UPDATE_ACCOUNT_ID))
    assert row, "probe account not in the optin update batch"
    payload = row_to_sf_record(row)
    acc_id = payload.pop("Id")
    print("payload:", payload)

    with SalesforceClientCC(load_salesforce_cc_config_from_env()) as sf:
        sf.authenticate()
        sf.update_account_by_id(acc_id, payload)
        back = sf.query(
            "SELECT Id, PersonEmail, InvestCustomer__pc, InvestmentStatus__pc, "
            "InvestmentExpirationDate__pc, LastModifiedDate "
            f"FROM Account WHERE Id = '{acc_id}'"
        )["records"][0]
        back.pop("attributes", None)
        for k, v in back.items():
            print(f"  {k:30s} {v}")
else:
    print("probe gated (RUN_PROBE_UPDATE = False)")

payload: {'InvestCustomer__pc': True}
  Id                             001Te00000ZqiFLIAZ
  PersonEmail                    lexer.sophia@gmail.com
  InvestCustomer__pc             True
  InvestmentStatus__pc           None
  InvestmentExpirationDate__pc   None
  LastModifiedDate               2026-08-28T13:08:43.000+0000


## 4. Probe: ONE person account insert

Single REST insert via `create_account_person` using the loader's own
`row_to_sf_record` (so probe == bulk payload). Afterwards the staging row gets
`sf_account_id`/`sf_person_contact_id` + `_account_processed_at` so the bulk load
skips it. Readback must prove `IsPersonAccount = true`. Set the external_id by hand.

### 4a. Payload preview (no Salesforce contact)

Builds the exact record the probe (and the bulk load, same mapper) will send for
`PROBE_INSERT_EXTERNAL_ID` and prints it. Fields that are NULL in staging are
absent here and stay untouched in the org. Review this before flipping
`RUN_PROBE_INSERT`.

In [ ]:
import json as _json

from insert_person_accounts_prereg import row_to_sf_record as _insert_payload

PROBE_INSERT_EXTERNAL_ID = "409c4d82-ae99-4914-9e20-f4b1c7165bfb"   # hand-entered, business-agreed

probe_row = db.fetch_one("""
    SELECT * FROM crm_imp_person_accounts
    WHERE _batch_id IN (%s, %s) AND external_id = %s
      AND _excluded = 0 AND _account_processed_at IS NULL
""", (*INSERT_BATCHES, PROBE_INSERT_EXTERNAL_ID))
assert probe_row, "probe row not found / already processed"
print(_json.dumps(_insert_payload(probe_row), indent=2, ensure_ascii=False, default=str))

In [ ]:
RUN_PROBE_INSERT = False   # flip only after reviewing the 4a payload preview
assert PROBE_INSERT_EXTERNAL_ID, "run the 4a preview cell first"

if RUN_PROBE_INSERT:
    from salesforce_client_prod import SalesforceClientCC, load_salesforce_cc_config_from_env
    from insert_person_accounts_prereg import row_to_sf_record

    row = db.fetch_one("""
        SELECT * FROM crm_imp_person_accounts
        WHERE _batch_id IN (%s, %s) AND external_id = %s
          AND _excluded = 0 AND _account_processed_at IS NULL
    """, (*INSERT_BATCHES, PROBE_INSERT_EXTERNAL_ID))
    assert row, "probe row not found / already processed"
    payload = row_to_sf_record(row)
    print("payload:", {k: v for k, v in payload.items()})

    with SalesforceClientCC(load_salesforce_cc_config_from_env()) as sf:
        sf.authenticate()
        acc_id = sf.create_account_person(payload)
        print("created:", acc_id)
        back = sf.query(
            "SELECT Id, IsPersonAccount, PersonContactId, PersonEmail, FirstName, LastName, "
            "InvestCustomer__pc, InvestmentStatus__pc, ExternalID__pc, RecordTypeId "
            f"FROM Account WHERE Id = '{acc_id}'"
        )["records"][0]
        back.pop("attributes", None)
        for k, v in back.items():
            print(f"  {k:20s} {v}")
        assert back["IsPersonAccount"] in (True, "true"), "NOT a person account - wrong RecordTypeId!"

        db.execute("""
            UPDATE crm_imp_person_accounts
            SET sf_account_id = %s, sf_person_contact_id = %s, _account_processed_at = NOW()
            WHERE row_id = %s
        """, (acc_id, back["PersonContactId"], row["row_id"]))
        print("staging row marked processed (bulk load will skip it)")
else:
    print("probe gated (RUN_PROBE_INSERT = False)")

## 5. Bulk load: account UPDATES

**Gated. Only after explicit go-ahead.** Loader aborts on duplicate sf_account_id,
reads `successfulResults`, writes `_processed_at` per Bulk batch.

In [ ]:
RUN_LOAD_UPDATE = False

if RUN_LOAD_UPDATE:
    for b in UPDATE_BATCHES:
        run_loader("update_accounts_prereg.py", b)
else:
    print("load gated (RUN_LOAD_UPDATE = False)")

## 6. Bulk load: account INSERTS

**Gated. Only after explicit go-ahead.** Loader matches back via `ExternalID__pc`,
writes `sf_account_id` + `_account_processed_at`, then fetches `PersonContactId`
per 200er chunk and writes `sf_person_contact_id`.

In [ ]:
RUN_LOAD_INSERT = False

if RUN_LOAD_INSERT:
    for b in INSERT_BATCHES:
        run_loader("insert_person_accounts_prereg.py", b)
else:
    print("load gated (RUN_LOAD_INSERT = False)")

## 7. Post-load verification (staging side)

Every non-excluded row processed; every insert row has both sf ids. A missing
`sf_person_contact_id` means a Business Account slipped through — stop and check
`logs/insert_person_accounts_prereg.log`.

In [ ]:
for b in UPDATE_BATCHES:
    r = db.fetch_one("""
        SELECT COUNT(*) AS n, SUM(_processed_at IS NULL AND _excluded = 0) AS still_open
        FROM crm_imp_person_accounts WHERE _batch_id = %s
    """, (b,))
    print(f"{b}: {int(r['n']):,} rows | still open: {int(r['still_open'] or 0)}")
    assert int(r["still_open"] or 0) == 0

for b in INSERT_BATCHES:
    r = db.fetch_one("""
        SELECT COUNT(*) AS n,
               SUM(_account_processed_at IS NULL AND _excluded = 0) AS still_open,
               SUM(_excluded = 0 AND sf_account_id IS NULL) AS no_acc,
               SUM(_excluded = 0 AND sf_person_contact_id IS NULL) AS no_pc
        FROM crm_imp_person_accounts WHERE _batch_id = %s
    """, (b,))
    print(f"{b}: {int(r['n']):,} rows | open: {int(r['still_open'] or 0)} | "
          f"no sf_account_id: {int(r['no_acc'] or 0)} | no PersonContactId: {int(r['no_pc'] or 0)}")
    assert int(r["still_open"] or 0) == 0
    assert int(r["no_pc"] or 0) == 0, "insert without PersonContactId - Business Account?"

print("account phase verified")

## 8. Mirror refresh + CPE backfill (prereq for the consent phase)

**Manual step first:** run `python camping-grubhof-import/refresh_sf_mirrors.py` at
the repo root, then re-apply `01_create_mirror_indexes.sql` (the refresh drops the
indexes). Only then execute the cells below.

The CPE is created by org automation on Person Account insert; nothing in this repo
creates one. Join is `PartyID__c = sf_person_contact_id AND EmailAddress = email` —
pass 2 retries the leftovers with the IDNA-encoded domain (Salesforce punycodes
non-ASCII domains on insert).

In [ ]:
row = db.fetch_one(
    "SELECT MAX(LastModifiedDate) AS newest, MAX(CreatedDate) AS newest_created "
    "FROM crm_cp_email_sfid_prod"
)
print(f"cpe mirror newest: {row['newest']} / created {row['newest_created']}")
# The refresh must postdate the insert load - otherwise the new CPEs are not in it.

dupes = db.fetch_all("""
    SELECT cpe.PartyID__c, cpe.EmailAddress, COUNT(*) AS n
    FROM crm_cp_email_sfid_prod cpe
    JOIN crm_imp_person_accounts pa
      ON pa.sf_person_contact_id = cpe.PartyID__c
    WHERE pa._batch_id LIKE %s
    GROUP BY cpe.PartyID__c, cpe.EmailAddress HAVING COUNT(*) > 1
""", (f"{BATCH_DATE}_prereg_%",))
print(f"duplicate CPEs on batch contacts: {len(dupes)}")
assert not dupes, "resolve duplicate CPEs before the backfill"


In [ ]:
updated = db.execute("""
    UPDATE crm_imp_person_accounts pa
    INNER JOIN crm_cp_email_sfid_prod cpe
            ON  cpe.PartyID__c   = pa.sf_person_contact_id
            AND cpe.EmailAddress = pa.email
    SET pa.sf_cp_email_id = cpe.Id
    WHERE pa._batch_id LIKE %s
      AND pa.sf_person_contact_id IS NOT NULL
      AND pa.sf_cp_email_id IS NULL
""", (f"{BATCH_DATE}_prereg_%",))
print(f"pass 1 (exact email): {updated:,} CPE ids backfilled")

# Pass 2: punycode domains
leftovers = db.fetch_all("""
    SELECT row_id, email, sf_person_contact_id
    FROM crm_imp_person_accounts
    WHERE _batch_id LIKE %s AND _excluded = 0
      AND sf_person_contact_id IS NOT NULL AND sf_cp_email_id IS NULL
      AND email IS NOT NULL
""", (f"{BATCH_DATE}_prereg_%",))
fixed = 0
for r in leftovers:
    local, _, domain = r["email"].rpartition("@")
    try:
        idna = f"{local}@{domain.encode('idna').decode()}"
    except UnicodeError:
        continue
    if idna == r["email"]:
        continue
    fixed += db.execute("""
        UPDATE crm_imp_person_accounts pa
        INNER JOIN crm_cp_email_sfid_prod cpe
                ON  cpe.PartyID__c   = pa.sf_person_contact_id
                AND cpe.EmailAddress = %s
        SET pa.sf_cp_email_id = cpe.Id
        WHERE pa.row_id = %s AND pa.sf_cp_email_id IS NULL
    """, (idna, r["row_id"]))
print(f"pass 2 (punycode): {fixed} of {len(leftovers)} leftovers backfilled")

In [ ]:
no_cpe = db.fetch_df("""
    SELECT _batch_id, row_id, external_id, email, sf_account_id, sf_person_contact_id
    FROM crm_imp_person_accounts
    WHERE _batch_id LIKE %s AND _excluded = 0 AND sf_cp_email_id IS NULL
""", (f"{BATCH_DATE}_prereg_%",))
out = REPO_ROOT / "local_data" / "prereg_no_cpe_after_backfill.csv"
no_cpe.to_csv(out, index=False)
print(f"{len(no_cpe):,} rows still without CPE -> {out}")
print("these rows CANNOT get a consent; exclude them for the consent phase in 05 "
      "or clarify manually (we never create CPEs ourselves)")

## Next

`05_run_prereg_consents.ipynb` — consent dry-runs, probe, load, verify, archive.
Do NOT start it before section 7 asserts pass and the CPE backfill is complete.